# DDM fit quality: psychometric, chronometric and no-response functions

Data vs model for each mouse, on control trials (no scheduled pulse). The model (`MODEL`) is fitted to all trials of each mouse
(`processed/ddm/<FITS>/<mouse>_<MODEL>_all.pkl`, from `scripts/ddm/fit_mouse_ddm.py`). The saved figure
`ddm_fit_quality_common_model.png` shows the earlier common model without guesses (`fits/..._L1U1V1Z1K0_all.pkl`). The model is simulated on the
mouse's own control trials (`N_REPS` per trial) with the same outcome rules as the fit: no response by 5 s is
censored, and decisions before 0.3 s register at 0.3 s.

- Psychometric: P(right) among responses. Chronometric: mean RT of responses (0.3-5 s), correct and error trials
  separately (0% coherence: all responses, plotted as correct); error points need >= 5 trials.
- Data: mean ± SE (binomial SE for proportions, SEM for RT). Model: lines.

In [ ]:
import logging
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scripts.ddm.fit_mouse_ddm import CENSOR_AT, LIKELIHOOD_PARAMS, MIN_RT, MouseDDM, load_inputs, make_params, select
from src.utils import scientific_plotting as spu

logging.disable(logging.WARNING)
spu.setup()
from config import dir_config

PROCESSED = Path(dir_config.data.processed)
MODEL = "L1U1V1Z0G1"  # leading candidate: leak, urgency, sv, fast guesses (no sz)
FITS = "fits_guess"
N_REPS = 20
OUT = Path(f"../dissemination/figures/ddm_fit_quality_{MODEL}.png")  # relative to notebooks/
OUT.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
inputs = load_inputs(PROCESSED / "ddm/ddm_inputs_global.npz")
mice = sorted(np.unique(inputs["subject_id"]), key=lambda m: (not m.startswith("PSUIM"), m))


def simulate_mouse(mouse):
    rows = np.flatnonzero((inputs["subject_id"] == mouse) & (inputs["sched_pulse_type"] == 0))
    rig = str(np.unique(inputs["data_source"][rows])[0])
    data, stim = select(inputs, rows)
    with open(PROCESSED / f"ddm/{FITS}/{mouse}_{MODEL}_all.pkl", "rb") as f:
        params = pickle.load(f)["parameters"]
    free, fixed = make_params(MODEL)
    model = MouseDDM(free, fixed, device="cpu", likelihood_params=LIKELIHOOD_PARAMS, censor_at=CENSOR_AT, min_rt=MIN_RT[rig])
    sim = model._map_outcomes(model.simulate(stim, params, n_reps=N_REPS, seed=0, labels=data["condition"]))
    coh = inputs["signed_coherence"][rows] * 100
    observed = pd.DataFrame({"coh": coh, "rt": data["rt"], "choice": data["choice"]})
    predicted = pd.DataFrame({"coh": coh[sim["trial"].to_numpy()], "rt": sim["rt"].to_numpy(), "choice": sim["choice"].to_numpy().astype(int)})
    return observed, predicted, rig


def summarize(df):
    responded = df[df.choice >= 0].copy()
    responded["correct"] = np.where(responded.coh == 0, np.nan, (responded.choice == 1) == (responded.coh > 0))
    psych = responded.groupby("coh").choice.agg(["mean", "count"])
    psych["se"] = np.sqrt(psych["mean"] * (1 - psych["mean"]) / psych["count"])
    responded["abs_coh"] = responded.coh.abs()
    chrono = {}
    for label, sub in [("correct", responded[(responded.correct == 1) | (responded.coh == 0)]), ("error", responded[responded.correct == 0])]:
        g = sub.groupby("abs_coh").rt.agg(["mean", "sem", "count"])
        chrono[label] = g[g["count"] >= 5]
    censored = df.assign(c=df.choice == -1).groupby("coh").c.agg(["mean", "count"])
    censored["se"] = np.sqrt(censored["mean"] * (1 - censored["mean"]) / censored["count"])
    return psych, chrono, censored

In [ ]:
fig, axes = plt.subplots(len(mice), 3, figsize=(spu.TEXTWIDTH * 1.25, 2.0 * len(mice)), sharex="col")
for row, mouse in enumerate(mice):
    observed, predicted, rig = simulate_mouse(mouse)
    (p_d, c_d, n_d), (p_m, c_m, n_m) = summarize(observed), summarize(predicted)
    ax = axes[row, 0]
    ax.errorbar(p_d.index, p_d["mean"], yerr=p_d["se"], fmt="o", color="k", ms=4, capsize=2, label="data")
    ax.plot(p_m.index, p_m["mean"], "-", color="#1a8fe3", lw=2, label="model")
    ax.set(ylim=(-0.02, 1.02), ylabel="P(right)")
    ax.set_title(f"{mouse} ({rig.upper()}, n={len(observed)})", fontsize=9, loc="left")
    ax = axes[row, 1]
    for label, color, marker in [("correct", "#2a9d3c", "o"), ("error", "#d11149", "s")]:
        d, m = c_d[label], c_m[label]
        ax.errorbar(d.index, d["mean"], yerr=d["sem"], fmt=marker, color=color, ms=4, capsize=2, mfc="white" if label == "error" else color, label=f"{label} (data)")
        ax.plot(m.index, m["mean"], "-", color=color, lw=2, label=f"{label} (model)")
    ax.set(ylabel="Mean RT (s)")
    ax = axes[row, 2]
    ax.errorbar(n_d.index, n_d["mean"], yerr=n_d["se"], fmt="o", color="k", ms=4, capsize=2)
    ax.plot(n_m.index, n_m["mean"], "-", color="#1a8fe3", lw=2)
    ax.set(ylim=(0, None), ylabel="P(no response by 5 s)")
axes[-1, 0].set_xlabel("Signed coherence (%)")
axes[-1, 1].set_xlabel("|Coherence| (%)")
axes[-1, 2].set_xlabel("Signed coherence (%)")
axes[0, 0].legend(fontsize=7, frameon=False)
axes[0, 1].legend(fontsize=6, frameon=False, ncol=2)
fig.suptitle(f"DDM {MODEL} vs data, control trials, mean ± SE", fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.985))
fig.savefig(OUT, dpi=110)
plt.show()

## Notes

- Fast responses (< 0.6 s) are 17% (UCLA) and 20% (UW) of responses and are near chance at every coherence (74% / 59%
  correct even at 100% coherence). They make up about 45% of errors at 100% coherence, so data error RTs at high
  coherence are much faster than correct RTs. The DDM cannot produce chance-level fast responses once the
  ~116 ms filter delay has passed, so it misses these fast errors. A fast-guess (contaminant) component is the
  candidate fix.